# Model 3 — 01 Data Validation & Provenance

## Purpose
Validate all CIC-IDS2017 shards and create reproducible dataset provenance.

This notebook does not train anything. It verifies the dataset before any model sees it and records row counts, labels, headers and SHA-256 hashes.

In [1]:

from pathlib import Path
import sys, json, os, warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "EVIDENT" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "behaviour"
MODEL_DIR = PROJECT_ROOT / "models" / "behavior"
ENV_DIR = MODEL_DIR / "environments"
META_DIR = MODEL_DIR / "metadata"
PROD_DIR = MODEL_DIR / "production"

for d in (ENV_DIR, META_DIR, PROD_DIR):
    d.mkdir(parents=True, exist_ok=True)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

import pandas as pd
import numpy as np

def clean_col(c):
    return " ".join(str(c).strip().replace("\ufeff","").split())

def norm_label(x):
    return "UNKNOWN" if pd.isna(x) else str(x).strip().upper().replace(" ","_")

def family(x):
    x = norm_label(x)
    if x == "BENIGN": return "BENIGN"
    if "PATATOR" in x or "BRUTE" in x: return "BRUTE_FORCE"
    if "PORTSCAN" in x or "PORT_SCAN" in x: return "PORT_SCAN"
    if "DDOS" in x or x.startswith("DOS"): return "DOS_DDOS"
    if "WEB_ATTACK" in x or "SQL_INJECTION" in x or "XSS" in x: return "WEB_ATTACK"
    if "BOT" in x: return "BOTNET"
    if "INFILTRATION" in x: return "INFILTRATION"
    if "HEARTBLEED" in x: return "HEARTBLEED"
    return "OTHER_ATTACK"

files = sorted(DATA_DIR.glob("*.csv"))
print("EVIDENT root:", PROJECT_ROOT)
print("CIC-IDS2017 files:", len(files))
for p in files:
    print(" -", p.name)
assert files, "No CIC-IDS2017 CSV files found."


EVIDENT root: c:\Users\bimal\OneDrive\Desktop\EVIDENT
CIC-IDS2017 files: 8
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [2]:

from collections import Counter
file_stats = []
overall = Counter()

for path in files:
    header = pd.read_csv(path, nrows=3, low_memory=False)
    header.columns = [clean_col(c) for c in header.columns]
    assert "Label" in header.columns, f"Label column missing: {path.name}"

    rows = 0
    local = Counter()
    for chunk in pd.read_csv(path, chunksize=100000, low_memory=False):
        chunk.columns = [clean_col(c) for c in chunk.columns]
        labels = chunk["Label"].map(norm_label)
        rows += len(chunk)
        local.update(labels.value_counts().to_dict())

    overall.update(local)
    file_stats.append({
        "name": path.name,
        "rows": rows,
        "labels": dict(local),
        "columns": list(header.columns)
    })
    print(f"{path.name}: {rows:,} rows")

provenance = {
    "dataset": "CIC-IDS2017",
    "total_rows": int(sum(x["rows"] for x in file_stats)),
    "files": file_stats,
    "labels": dict(overall),
}
(META_DIR/"dataset_provenance.json").write_text(
    json.dumps(provenance, indent=2, default=str), encoding="utf-8"
)
print("\nOverall labels:")
print(json.dumps(dict(overall), indent=2))
print("Saved:", META_DIR/"dataset_provenance.json")


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
Monday-WorkingHours.pcap_ISCX.csv: 529,918 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows
Wednesday-workingHours.pcap_ISCX.csv: 692,703 rows

Overall labels:
{
  "DDOS": 128027,
  "BENIGN": 2273097,
  "PORTSCAN": 158930,
  "BOT": 1966,
  "INFILTRATION": 36,
  "WEB_ATTACK_\ufffd_BRUTE_FORCE": 1507,
  "WEB_ATTACK_\ufffd_XSS": 652,
  "WEB_ATTACK_\ufffd_SQL_INJECTION": 21,
  "FTP-PATATOR": 7938,
  "SSH-PATATOR": 5897,
  "DOS_HULK": 231073,
  "DOS_SLOWLORIS": 5796,
  "DOS_SLOWHTTPTEST": 5499,
  "DOS_GOLDENEYE": 10293,
  "HEARTBLEED": 11
}
Saved: c:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\dataset_provenance.json
